In [ ]:
# 1. System packages & uv setup
import subprocess
import sys
import os

print('Installing system packages: espeak-ng, ffmpeg...')
subprocess.run(['apt-get', 'update', '-qq'], check=False)
subprocess.run(['apt-get', 'install', '-qq', '-y', 'espeak-ng', 'ffmpeg'], check=False)

print('Installing uv...')
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'uv'])

print('Creating Python 3.11 environment with uv...')
subprocess.check_call(['uv', 'venv', '--python', '3.11', '/tmp/audio_env'])
py_bin = '/tmp/audio_env/bin/python'

print('Installing kokoro, soundfile, pydub, faster-whisper and en_core_web_sm into Python 3.11...')
subprocess.check_call([
    'uv', 'pip', 'install', '--python', py_bin,
    'torch', 'kokoro>=0.9.2', 'soundfile', 'pydub', 'faster-whisper',
'nvidia-cublas-cu12', 'nvidia-cudnn-cu12',
    'https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl'
])
print('Dependencies installed successfully into Python 3.11.')


In [ ]:
# 2. Script definition
text = "You\u2019re one habit away from changing everything. Not five habits, not a thirty-day challenge, not waking up at four in the morning, just one, but it\u2019s not the one you think.\n\nMost people try to change their life by adding motivation, and that usually fails. The real shift happens when you build one habit that controls other habits. That habit is daily non-negotiable output. Every single day, you must produce something, not consume, produce.\n\nWhat counts as output? One page written, one workout done, one skill practiced, one sales call made, one video edited, one idea executed. Small is fine, but zero is not.\n\nHere\u2019s why this habit changes everything. First, it kills overthinking. You stop planning your life at two in the morning and you start building it at two in the afternoon. Second, it builds proof. Confidence doesn\u2019t come from affirmations. It comes from evidence. Third, it compounds. Three hundred and sixty-five small outputs create an unfair advantage. And fourth, it rewires your identity. You stop saying, \u201cI want to be disciplined.\u201d And you become someone who produces daily.\n\nMost people\u2019s problems aren\u2019t a lack of talent. It\u2019s infinite consumption. Scrolling, watching, learning, saving posts, planning, and constantly restarting. You don\u2019t need more information. You need one rule. No day without output. Even on bad days, especially on bad days.\n\nThe scary part is that if you do this for one year, your life will look completely different. Not because of motivation, but because of volume.\n\nNow ask yourself honestly, what is your daily non-negotiable output going to be?"

with open('script_text.txt', 'w', encoding='utf-8') as f:
    f.write(text)

print(f'Script loaded: {len(text)} characters')
print(text[:200] + ('...' if len(text) > 200 else ''))


In [ ]:
# 3. Execution pipeline in Python 3.11
runner_code = "\nimport os\nimport json\nimport torch\nimport numpy as np\nimport soundfile as sf\nfrom pydub import AudioSegment\nfrom kokoro import KPipeline\nfrom faster_whisper import WhisperModel\n\nwith open(\"script_text.txt\", \"r\", encoding=\"utf-8\") as f:\n    text = f.read().strip()\n\nprint(f\"Synthesizing audio for {len(text)} characters...\")\nprint('Initializing Kokoro-82M KPipeline(lang_code=\"a\")...')\npipeline = KPipeline(lang_code=\"a\")\n\nprint(\"Synthesizing audio with voice='am_michael' at speed=1.0...\")\ngenerator = pipeline(\n    text,\n    voice=\"am_michael\",\n    speed=1.0,\n    split_pattern=r\"\\n+\"\n)\n\nall_audio = []\nfor i, (gs, ps, audio) in enumerate(generator):\n    if audio is not None:\n        if isinstance(audio, torch.Tensor):\n            audio = audio.detach().cpu().numpy()\n        all_audio.append(audio)\n\nif not all_audio:\n    raise RuntimeError(\"Kokoro produced no audio chunks.\")\n\ncombined_audio = np.concatenate(all_audio, axis=0)\nsf.write(\"audio.wav\", combined_audio, 24000)\n\naudio_seg = AudioSegment.from_wav(\"audio.wav\")\naudio_seg.export(\"audio.mp3\", format=\"mp3\", bitrate=\"192k\")\nprint(\"Kokoro-82M audio synthesis complete: audio.mp3\")\n\ndevice = \"cuda\" if torch.cuda.is_available() else \"cpu\"\ncompute_type = \"float16\" if device == \"cuda\" else \"int8\"\nprint(f\"Loading Whisper model on {device} ({compute_type})...\")\n\ntry:\n    model = WhisperModel(\"base.en\", device=device, compute_type=compute_type)\n    segments, info = model.transcribe(\"audio.mp3\", word_timestamps=True)\n    words_data = []\n    for segment in segments:\n        for word in segment.words:\n            clean_word = word.word.strip()\n            if clean_word:\n                words_data.append({\n                    \"word\": clean_word,\n                    \"start\": round(word.start, 3),\n                    \"end\": round(word.end, 3)\n                })\nexcept Exception as e:\n    print(f\"CUDA whisper failed ({e}), falling back to CPU...\")\n    model = WhisperModel(\"base.en\", device=\"cpu\", compute_type=\"int8\")\n    segments, info = model.transcribe(\"audio.mp3\", word_timestamps=True)\n    words_data = []\n    for segment in segments:\n        for word in segment.words:\n            clean_word = word.word.strip()\n            if clean_word:\n                words_data.append({\n                    \"word\": clean_word,\n                    \"start\": round(word.start, 3),\n                    \"end\": round(word.end, 3)\n                })\n\nOUTPUT_TIMESTAMPS = \"timestamps.json\"\nwith open(OUTPUT_TIMESTAMPS, \"w\", encoding=\"utf-8\") as f:\n    json.dump(words_data, f, indent=2)\n\nprint(f\"Saved {len(words_data)} word timestamps to {OUTPUT_TIMESTAMPS}.\")\nprint(\"All audio and alignment steps finished successfully!\")\n"

with open('pipeline_runner.py', 'w', encoding='utf-8') as f:
    f.write(runner_code)

import subprocess, os
py_bin = '/tmp/audio_env/bin/python'
env = os.environ.copy()
env['PATH'] = '/tmp/audio_env/bin:' + env.get('PATH', '')
env['VIRTUAL_ENV'] = '/tmp/audio_env'
print('Executing audio pipeline in isolated Python 3.11 environment...')
subprocess.check_call([py_bin, 'pipeline_runner.py'], env=env)
print('Notebook execution finished successfully!')
